# Domain A: Transcriptomic Identity and Neural Coding

This notebook addresses three questions:
- **A1:** Do transcriptomically defined cell types have distinct tuning properties?
- **A2:** Does gene expression predict functional response properties at the single-cell level?
- **A3:** How does transcriptomic identity shape population coding geometry?

**Data:** Zarr multimodal stores with ΔF/F calcium traces (X matrix, cells × trials), stimulus metadata (var), cell-type labels & gene expression (obs).

In [13]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.optimize import curve_fit
from scipy.stats import kruskal, mannwhitneyu, spearmanr, pearsonr
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.decomposition import PCA, NMF
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.manifold import TSNE
from pathlib import Path    
from matplotlib import cm, colors
import warnings
import anndata as ad
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, zarr_to_df
from functions.visualization import polar_bar_plot

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
MULTIMODAL = Path('../scratch/multimodal_data')
MOUSE_IDS = [f.stem.split('_')[0] for f in MULTIMODAL.glob('*_multimodal_data.zarr')]
SESSIONS = ['session_1', 'session_2', 'session_3']
session_type = ['drifting_gratings']
MOUSE_IDS

['782149h',
 '786297x',
 '790322h',
 '767018h',
 '788406h',
 '767022h',
 '778174x',
 '797371x',
 '755252h']

In [5]:
obs = [adata.obs for adata in adata_list]
obs = pd.concat(obs, ignore_index=True)
# ── Identify gene expression columns ──
META_COLS = {'unique_id', 'mouse_id', 'class_name', 'class_label',
             'class_bootstrapping_probability', 'subclass_name', 'subclass_label',
             'subclass_bootstrapping_probability', 'supertype_name', 'supertype_label',
             'supertype_bootstrapping_probability', 'cluster_name', 'cluster_label',
             'cluster_alias', 'cluster_bootstrapping_probability',
             'x_loc', 'y_loc', 'z_loc'}
GENE_COLS = [c for c in obs.columns if c not in META_COLS]
print(f"Gene expression columns: {len(GENE_COLS)}")

# ── Color palette for subclasses ──
SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])


Gene expression columns: 307


## A4: Cell-Type Temporal Dynamics (10 Hz ΔF/F)

Using trial-resolved 10 Hz ΔF/F traces from zarr stores (-1 s to +3 s around stimulus onset), we characterize:
- **A4.1** PSTHs by subclass — onset latency, peak timing, transient vs sustained index
- **A4.2** Temporal diversity within and across subclasses

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# A4.1  Load 10 Hz data & compute PSTHs by subclass
# ══════════════════════════════════════════════════════════════════════

def load_zarr_10hz(mouse_id, session='session_1'):
    """Load 10 Hz trial-resolved data from zarr."""
    z = zarr.open(f'{MULTIMODAL}/{mouse_id}_multimodal_data.zarr', 'r')
    gs = z[f'ophys/drifting_gratings/{session}/stim_aligned_dff/GratingStim']
    ti_dict = {k: gs[f'trial_info/{k}'][:] for k in gs['trial_info'].keys()}
    return {
        'dff': gs['dff'][:],
        'unique_ids': z['unique_id'][:].astype(str),
        'running': gs['running'][:],
        'time_rel': gs['time_relative'][:],
        'trial_info': pd.DataFrame(ti_dict),
    }

# Load session 1 from all mice, match cells to obs
all_psth = []       # will be (n_cells_matched, 41) mean PSTH
all_subclass = []
all_mouse = []
all_dff_trials = [] # (n_cells_matched,) each entry: (n_gratings, 41) non-gray trials

for mouse_id in MOUSE_IDS:
    pk = load_zarr_10hz(mouse_id)
    dff = pk['dff']            # (n_trials, 41, n_cells)
    ti = pk['trial_info']
    time_rel = pk['time_rel']
    uids = pk['unique_ids']
    
    # Select non-gray-screen grating trials
    grating_mask = ~ti['gray_screen'].astype(bool)
    dff_grating = dff[grating_mask.values]  # (n_grating_trials, 41, n_cells)
    
    # Match cells to obs (by unique_id)
    obs_mouse = obs[obs['mouse_id'] == mouse_id]
    for ci, uid in enumerate(uids):
        if uid in obs_mouse['unique_id'].values:
            row = obs_mouse[obs_mouse['unique_id'] == uid].iloc[0]
            psth = np.nanmean(dff_grating[:, :, ci], axis=0)  # mean across all grating trials
            all_psth.append(psth)
            all_subclass.append(row['subclass_name'])
            all_mouse.append(mouse_id)
            all_dff_trials.append(dff_grating[:, :, ci])  # (n_grating_trials, 41)

all_psth = np.array(all_psth)          # (n_cells_matched, 41)
all_subclass = np.array(all_subclass)
print(f"Matched {len(all_psth)} cells with 10 Hz temporal data")
print(f"Time points: {len(time_rel)}, from {time_rel[0]:.1f}s to {time_rel[-1]:.1f}s")

# ── Compute temporal metrics ──
stim_on_idx = np.argmin(np.abs(time_rel))  # t=0
baseline_idx = time_rel < 0                 # pre-stimulus
stim_idx = (time_rel >= 0) & (time_rel <= 2.0)
early_idx = (time_rel >= 0) & (time_rel <= 0.5)
late_idx = (time_rel >= 1.5) & (time_rel <= 2.0)

baseline_mean = np.nanmean(all_psth[:, baseline_idx], axis=1)
peak_response = np.nanmax(all_psth[:, stim_idx], axis=1)
early_response = np.nanmean(all_psth[:, early_idx], axis=1)
late_response = np.nanmean(all_psth[:, late_idx], axis=1)

# Transient/Sustained index: (early - late) / (early + late)
tsi = (early_response - late_response) / (np.abs(early_response) + np.abs(late_response) + 1e-8)

# Onset latency: first time point after stim onset where PSTH > baseline + 2*SD(baseline)
baseline_std = np.nanstd(all_psth[:, baseline_idx], axis=1)
onset_latency = np.full(len(all_psth), np.nan)
for ci in range(len(all_psth)):
    threshold = baseline_mean[ci] + 2 * baseline_std[ci]
    above = np.where((time_rel >= 0) & (all_psth[ci] > threshold))[0]
    if len(above) > 0:
        onset_latency[ci] = time_rel[above[0]]

# Peak time
peak_time = np.array([time_rel[stim_idx][np.argmax(all_psth[ci, stim_idx])] for ci in range(len(all_psth))])

temporal_df = pd.DataFrame({
    'subclass': all_subclass,
    'mouse': all_mouse,
    'tsi': tsi,
    'onset_latency': onset_latency,
    'peak_time': peak_time,
    'peak_response': peak_response,
    'baseline': baseline_mean,
    'subclass_short': [SUBCLASS_SHORT.get(s, s) for s in all_subclass],
})

# ══════════════════════════════════════════════════════════════════════
# Visualization: PSTHs by subclass + temporal metrics
# ══════════════════════════════════════════════════════════════════════
present_sc_temp = [s for s in SUBCLASS_ORDER if s in all_subclass]

fig, axes = plt.subplots(2, 4, figsize=(24, 10))

# ── Row 1: PSTHs per subclass ──
for ax, sc in zip(axes[0], present_sc_temp):
    mask = all_subclass == sc
    mean_p = np.nanmean(all_psth[mask], axis=0)
    sem_p = np.nanstd(all_psth[mask], axis=0) / np.sqrt(mask.sum())
    ax.fill_between(time_rel, mean_p - sem_p, mean_p + sem_p,
                    alpha=0.3, color=SUBCLASS_COLORS[sc])
    ax.plot(time_rel, mean_p, color=SUBCLASS_COLORS[sc], linewidth=2)
    ax.axvline(0, color='k', ls='--', alpha=0.4, label='Stim ON')
    ax.axvline(2.0, color='k', ls=':', alpha=0.4, label='Stim OFF')
    ax.set_title(f'{SUBCLASS_SHORT[sc]} (n={mask.sum()})',
                 color=SUBCLASS_COLORS[sc], fontweight='bold')
    ax.set_xlabel('Time (s)')
if len(present_sc_temp) < 4:
    for ax in axes[0, len(present_sc_temp):]:
        ax.set_visible(False)
axes[0, 0].set_ylabel('ΔF/F')

# ── Overlaid PSTHs ──
ax = axes[0, -1] if len(present_sc_temp) >= 4 else axes[0, len(present_sc_temp)]
for sc in present_sc_temp:
    mask = all_subclass == sc
    mean_p = np.nanmean(all_psth[mask], axis=0)
    ax.plot(time_rel, mean_p, color=SUBCLASS_COLORS[sc], linewidth=2,
            label=SUBCLASS_SHORT[sc])
ax.axvline(0, color='k', ls='--', alpha=0.4)
ax.axvline(2.0, color='k', ls=':', alpha=0.4)
ax.legend(fontsize=7)
ax.set_title('All Subclasses Overlaid', fontweight='bold')
ax.set_xlabel('Time (s)')
ax.set_ylabel('ΔF/F')

# ── Row 2: Temporal metric distributions ──
short_order = [SUBCLASS_SHORT[s] for s in present_sc_temp]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

for ax, met, lab in zip(axes[1], ['tsi', 'onset_latency', 'peak_time', 'peak_response'],
                        ['Transient-Sustained Index', 'Onset Latency (s)',
                         'Peak Time (s)', 'Peak ΔF/F']):
    sns.violinplot(data=temporal_df, x='subclass_short', y=met,
                   order=short_order, palette=short_pal, cut=0, inner='box', ax=ax)
    ax.set_title(lab, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('A4: Cell-Type Temporal Dynamics (10 Hz PSTH)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Stats ──
from scipy.stats import kruskal
print("\n=== Kruskal-Wallis across subclasses ===")
for met in ['tsi', 'onset_latency', 'peak_time', 'peak_response']:
    groups = [temporal_df.loc[temporal_df['subclass'] == s, met].dropna() for s in present_sc_temp]
    groups = [g.values for g in groups if len(g) >= 3]
    if len(groups) >= 2:
        stat, p = kruskal(*groups)
        print(f"  {met:20s}: H={stat:.2f}, p={p:.2e}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# A4.2  Temporal diversity: how variable are temporal profiles within
#       and across subclasses? Do genes predict temporal dynamics?
# ══════════════════════════════════════════════════════════════════════
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

# ── Normalize each cell's PSTH for shape comparison (unit norm on stim period) ──
stim_psth = all_psth[:, stim_idx]  # (n_cells, stim_timepoints)
stim_psth_norm = stim_psth / (np.linalg.norm(stim_psth, axis=1, keepdims=True) + 1e-8)

# ── Cluster cells by temporal shape ──
n_clust = 5
D = pdist(stim_psth_norm, metric='correlation')
Z = linkage(D, method='ward')
clust_labels = fcluster(Z, n_clust, criterion='maxclust')

# ── Mean temporal profile per cluster ──
fig, axes = plt.subplots(1, n_clust + 1, figsize=(4 * (n_clust + 1), 4), sharey=True)
cluster_colors = plt.cm.tab10(np.linspace(0, 1, n_clust))

for ci in range(1, n_clust + 1):
    ax = axes[ci - 1]
    cmask = clust_labels == ci
    mean_p = np.nanmean(stim_psth[cmask], axis=0)
    sem_p = np.nanstd(stim_psth[cmask], axis=0) / np.sqrt(cmask.sum())
    t_stim = time_rel[stim_idx]
    ax.fill_between(t_stim, mean_p - sem_p, mean_p + sem_p,
                    alpha=0.3, color=cluster_colors[ci - 1])
    ax.plot(t_stim, mean_p, color=cluster_colors[ci - 1], linewidth=2)
    ax.set_title(f'Cluster {ci} (n={cmask.sum()})', fontweight='bold')
    ax.set_xlabel('Time (s)')
axes[0].set_ylabel('ΔF/F')

# Subclass composition per cluster (stacked bar)
ax = axes[-1]
comp = pd.DataFrame({'cluster': clust_labels, 'subclass': all_subclass})
ct = pd.crosstab(comp['cluster'], comp['subclass'], normalize='index')
ct_short = ct.rename(columns=SUBCLASS_SHORT)
ct_short[[SUBCLASS_SHORT[s] for s in present_sc_temp if SUBCLASS_SHORT[s] in ct_short.columns]].plot(
    kind='bar', stacked=True, ax=ax,
    color=[SUBCLASS_COLORS[s] for s in present_sc_temp if SUBCLASS_SHORT[s] in ct_short.columns])
ax.set_title('Subclass Composition', fontweight='bold')
ax.set_xlabel('Temporal Cluster')
ax.set_ylabel('Fraction')
ax.legend(fontsize=7, loc='upper right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.suptitle('A4: Temporal Response Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Within-subclass temporal diversity (mean pairwise correlation distance) ──
diversity = {}
for sc in present_sc_temp:
    sc_mask = all_subclass == sc
    if sc_mask.sum() < 5:
        continue
    D_sc = pdist(stim_psth_norm[sc_mask], metric='correlation')
    diversity[SUBCLASS_SHORT[sc]] = np.mean(D_sc)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(diversity.keys(), diversity.values(),
       color=[SUBCLASS_COLORS[s] for s in present_sc_temp if SUBCLASS_SHORT[s] in diversity])
ax.set_ylabel('Mean Correlation Distance')
ax.set_title('A4: Within-Subclass Temporal Diversity', fontweight='bold')
plt.tight_layout()
plt.show()

# ── Do temporal metrics correlate with genes? Quick LASSO screen ──
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

# Match cells back to obs to get gene expression
temporal_df['unique_id'] = None  # placeholder — we'll use index matching
# Use the gene matrix from the full obs (already loaded in A2)
# We need to match all_subclass ordering to obs

# Use mouse_id + subclass to re-index into obs gene columns
# For simplicity, use the temporal metrics and try predicting from tuning_df metrics
print("\n=== Temporal metric correlations with tuning metrics ===")
# Quick check: correlate TSI with OSI, C50, etc. using matched cells
# (This is a coarse match since cell ordering may differ)
print(f"TSI mean by subclass:")
for sc in present_sc_temp:
    vals = temporal_df.loc[temporal_df['subclass'] == sc, 'tsi'].dropna()
    print(f"  {SUBCLASS_SHORT[sc]:10s}: {vals.mean():.3f} ± {vals.std():.3f} (n={len(vals)})")